# One IntelliSense, Four Objects

An object is a space, a set of named method families spanning subspaces of it, and a
target. Everything else — arenas, stoichiometry, series, bit mixing — is a constructor.

The five calls are the same in every domain:

| call | question |
|---|---|
| `methods()` | what is actually in this object |
| `signature(f)` | what does this method reach, what does it leave open |
| `required(*declared)` | what does this position still need |
| `candidates(*declared)` | which available methods close it — answered by evaluating |
| `minimal_vocabularies()` | the smallest sets that close the object |

Four instantiations follow. They share no machinery below the core: a configuration
space, a reaction network, a series identity, and a block cipher's linear layer. Two are
over ℚ and one is over GF(2). Each number below is checkable by hand.

In [1]:
"""IntelliSense for any object whose methods span a shared space.
Exact arithmetic: Fraction over Q, or GF(2)."""
from fractions import Fraction
from itertools import combinations

class GF2:
    name="GF(2)"
    @staticmethod
    def z(x): return int(x)&1
    @staticmethod
    def add(a,b): return (a^b)&1
    @staticmethod
    def mul(a,b): return (a&b)&1
    @staticmethod
    def inv(a): return 1
class QQ:
    name="Q"
    @staticmethod
    def z(x): return Fraction(x)
    @staticmethod
    def add(a,b): return a+b
    @staticmethod
    def mul(a,b): return a*b
    @staticmethod
    def inv(a): return 1/a

def rref(rows, nc, F=QQ):
    M=[[F.z(x) for x in r] for r in rows]; piv=[]; rr=0
    for c in range(nc):
        p=next((i for i in range(rr,len(M)) if M[i][c]!=0), None)
        if p is None: continue
        M[rr],M[p]=M[p],M[rr]
        iv=F.inv(M[rr][c]); M[rr]=[F.mul(x,iv) for x in M[rr]]
        for i in range(len(M)):
            if i!=rr and M[i][c]!=0:
                f=M[i][c]
                M[i]=[F.add(a, F.mul(-f,b) if F is QQ else F.mul(f,b))
                      for a,b in zip(M[i],M[rr])]
        piv.append(c); rr+=1
        if rr==len(M): break
    return M[:rr], piv
def rank(rows, nc, F=QQ): return 0 if not rows else len(rref(rows,nc,F)[0])
def inspan(v, rows, nc, F=QQ):
    return rank(rows,nc,F)==rank(list(rows)+[v],nc,F)

class Obj:
    """An object: a space, named method families spanning subspaces, and a target.
    The target defaults to the whole space (then 'closing' means saturating it)."""
    def __init__(self, dim, methods, name="object", F=QQ, target=None, units="",
                 capacity=None):
        self.dim, self.F, self.name, self.units = dim, F, name, units
        self.raw = {k:list(v) for k,v in methods.items() if v}
        self.b   = {k: rref(v,dim,F)[0] for k,v in self.raw.items()}
        self.target = target                     # None = saturate the capacity
        self.cap = (capacity if capacity is not None else dim) if target is None else 1

    def methods(self): return sorted(self.b, key=lambda k:-len(self.raw[k]))

    def _rows(self,fams): return [r for f in fams for r in self.b.get(f,[])]

    def signature(self,*fams):
        rows=self._rows(fams); rk=rank(rows,self.dim,self.F)
        if self.target is None:
            open_=self.cap-rk; reach=rk; cap=self.cap
        else:
            open_= 0 if inspan(self.target,rows,self.dim,self.F) else 1
            reach=rk; cap=1
        return {"families":tuple(fams),"ops":sum(len(self.raw.get(f,[])) for f in fams),
                "rank":reach,"capacity":cap,"open":open_}

    def required(self,*fams): return self.signature(*fams)["open"]

    def candidates(self,*fams):
        have=set(fams); out=[]
        for f in self.methods():
            if f in have: continue
            s=self.signature(*(tuple(have)+(f,)))
            out.append({"method":f,"ops":len(self.raw[f]),
                        "closes":s["open"]==0,"remaining":s["open"]})
        return sorted(out,key=lambda d:(not d["closes"],d["remaining"],d["ops"]))

    def minimal_vocabularies(self):
        ms=self.methods(); sat=[]
        for k in range(1,len(ms)+1):
            for s in combinations(ms,k):
                if self.signature(*s)["open"]==0: sat.append(s)
            if sat: break
        return sat

    def report(self):
        tgt = "saturate the space" if self.target is None else "reach the target"
        L=[f"OBJECT  {self.name}    over {self.F.name}    dim {self.dim}"
           f"    goal: {tgt}"]
        for f in self.methods():
            s=self.signature(f)
            L.append(f"   {f:<22s}{s['ops']:>6d} {self.units or 'ops':<10s}"
                     f" rank {s['rank']:>3d}   leaves {s['open']:>3d} open")
        a=self.signature(*self.methods())
        L.append(f"   {'ALL':<22s}{'':>6s} {'':<10s} rank {a['rank']:>3d}   "
                 + ("CLOSES" if a['open']==0 else f"leaves {a['open']} open"))
        mv=self.minimal_vocabularies()
        if mv: L.append("   minimal closing vocabularies: "+"; ".join("+".join(s) for s in mv))
        else:  L.append("   NO vocabulary in this object closes it")
        return "\n".join(L)


In [2]:
"""Four objects, one IntelliSense. Each constructor returns an Obj."""
from fractions import Fraction
from itertools import combinations

# ---------------------------------------------------------------- 1. token arena
def arena_obj(n, edges, K=2, group=None, name="arena"):
    Eset={frozenset(e) for e in edges}
    if group is None:
        adj=[set() for _ in range(n)]
        for a,b in edges: adj[a].add(b); adj[b].add(a)
        deg=[len(adj[v]) for v in range(n)]
        sig=[(deg[v],tuple(sorted(deg[u] for u in adj[v]))) for v in range(n)]
        order=sorted(range(n),key=lambda v:(-deg[v],v)); group=[]; p=[-1]*n; used=[False]*n
        def bt(k):
            if k==n: group.append(tuple(p)); return
            v=order[k]
            for w in range(n):
                if used[w] or sig[w]!=sig[v]: continue
                if all((u in adj[v])==(p[u] in adj[w]) for u in order[:k]):
                    p[v]=w; used[w]=True; bt(k+1); used[w]=False; p[v]=-1
        bt(0)
    V=sorted(combinations(range(n),K)); vi={frozenset(s):i for i,s in enumerate(V)}
    ae=set()
    for i,S in enumerate(V):
        s=set(S)
        for a in s:
            for b in range(n):
                if b in s or frozenset((a,b)) not in Eset: continue
                j=vi[frozenset((s-{a})|{b})]; ae.add((min(i,j),max(i,j)))
    ae=sorted(ae); act=lambda g,i: vi[frozenset(g[x] for x in V[i])]
    lab={}; bas=[]
    for oe in [(a,b) for a,b in ae]+[(b,a) for a,b in ae]:
        if oe in lab: continue
        orb={(act(g,oe[0]),act(g,oe[1])) for g in group}
        if (oe[1],oe[0]) in orb:
            for x in orb: lab[x]=(None,0)
        else:
            c=len(bas); bas.append(oe)
            for x in orb: lab[x]=(c,1)
            for x in orb: lab[(x[1],x[0])]=(c,-1)
    d=len(bas)
    adjA={}
    for a,b in ae: adjA.setdefault(a,set()).add(b); adjA.setdefault(b,set()).add(a)
    def cyc(L):
        out=set()
        def walk(s,c,p):
            if len(p)==L:
                if s in adjA.get(c,()) and p[1]<p[-1]: out.add(tuple(p)); return
            if len(p)>=L: return
            for nx in sorted(adjA.get(c,())):
                if nx<=s or nx in p: continue
                walk(s,nx,p+[nx])
        for s in sorted(adjA): walk(s,s,[s])
        return [list(c) for c in out]
    def row(c):
        v=[0]*d
        for i in range(len(c)):
            cc,s=lab[(c[i],c[(i+1)%len(c)])]
            if cc is not None: v[cc]+=s
        return v
    def mv(c):
        S=[set(V[x]) for x in c]
        return [frozenset(((S[i]-S[(i+1)%len(c)]).pop(),(S[(i+1)%len(c)]-S[i]).pop()))
                for i in range(len(c))]
    F={}
    for c in cyc(3):
        S=[set(V[x]) for x in c]
        F.setdefault("spec3" if set.intersection(*S) else "rot3",[]).append(row(c))
    for c in cyc(4):
        S=[set(V[x]) for x in c]
        k="disj" if len(set(mv(c)))==2 else ("spec4" if set.intersection(*S) else "rot4")
        F.setdefault(k,[]).append(row(c))
    # quotient by im d0 is implicit: cell rows already annihilate it, so rank d1 is
    # measured in the full invariant space and capacity is dim - rank d0
    seen=set(); d0=[]
    for i in range(len(V)):
        if i in seen: continue
        orb={act(g,i) for g in group}; seen|=orb
        d0.append([(1 if b in orb else 0)-(1 if a in orb else 0) for a,b in bas])
    D=d-rank(d0,d)
    return Obj(d,F,name=name,units="cells",capacity=D)

# ---------------------------------------------------------------- 2. reaction network
def crn_obj(species, reactions, name="reaction network"):
    """Methods are reactions; the space is species space; what stays open are the
    conservation laws the reaction set cannot change."""
    idx={s:i for i,s in enumerate(species)}
    meth={}
    for rn,(lhs,rhs) in reactions.items():
        v=[0]*len(species)
        for s,c in lhs.items(): v[idx[s]]-=c
        for s,c in rhs.items(): v[idx[s]]+=c
        meth[rn]=[v]
    return Obj(len(species),meth,name=name,units="reactions")

# ---------------------------------------------------------------- 3. BBP
def bbp_obj(groups, target, name="BBP base-16, degree 1"):
    """The space is the coefficient space over residues 1..8 of the base-16 BBP
    generators S_r = sum_k 16^-k/(8k+r). A constant is expressible from a set of
    residues iff its coefficient vector lies in their span."""
    meth={g:[[1 if j==r-1 else 0 for j in range(8)] for r in rs] for g,rs in groups.items()}
    return Obj(8,meth,name=name,target=[Fraction(x) for x in target],units="residues")

# ---------------------------------------------------------------- 4. GF(2) bit mixing
def _rotr(n,w=32): return [[1 if (j-i)%w==(w-n)%w else 0 for j in range(w)] for i in range(w)]
def _shr(n,w=32):  return [[1 if (j==i+n and i+n<w) else 0 for j in range(w)] for i in range(w)]
def _xor(*Ms):
    w=len(Ms[0]); return [[sum(M[i][j] for M in Ms)&1 for j in range(w)] for i in range(w)]
def sha_obj(name="SHA-256 bit-mixing (GF(2)-linear part)"):
    s0=_xor(_rotr(7),_rotr(18),_shr(3))
    s1=_xor(_rotr(17),_rotr(19),_shr(10))
    S0=_xor(_rotr(2),_rotr(13),_rotr(22))
    S1=_xor(_rotr(6),_rotr(11),_rotr(25))
    return Obj(32,{"sigma0":s0,"sigma1":s1,"Sigma0":S0,"Sigma1":S1},
               name=name,F=GF2,units="rows")


## 1. Token arena — cells in a configuration space

Methods are the relation families the arena contains. The capacity is the invariant
cochain dimension modulo gradients.

In [3]:
W = (12, [(0,1),(0,2),(0,4),(0,5),(0,6),(0,7),(0,9),(1,2),(1,3),(1,5),(1,7),(1,8),
          (1,10),(2,3),(2,4),(2,8),(2,9),(2,11),(3,4),(3,5),(3,6),(3,9),(3,10),
          (4,5),(4,7),(4,10),(4,11),(5,6),(5,8),(5,11)])
a = arena_obj(*W, K=2, name="token arena: 12-vertex witness, K=2")
print(a.report())
print("\ndeclared disj — requires", a.required('disj'), "dimensions:")
for c in a.candidates('disj'):
    print(f"   {c['method']:<7s} {c['ops']:>4d} cells  closes={c['closes']}  leaves {c['remaining']}")

OBJECT  token arena: 12-vertex witness, K=2    over Q    dim 22    goal: saturate the space
   spec4                    528 cells      rank   8   leaves   7 open
   disj                     291 cells      rank  13   leaves   2 open
   rot4                     264 cells      rank   7   leaves   8 open
   spec3                    180 cells      rank  10   leaves   5 open
   rot3                      20 cells      rank   2   leaves  13 open
   ALL                                     rank  15   CLOSES
   minimal closing vocabularies: disj+spec3; disj+rot3

declared disj — requires 2 dimensions:
   rot3      20 cells  closes=True  leaves 0
   spec3    180 cells  closes=True  leaves 0
   rot4     264 cells  closes=False  leaves 1
   spec4    528 cells  closes=False  leaves 1


Two of four close it; the cheapest is 20 cells against 180. The other two have the
reach and fall one dimension short — a signature is not a behaviour.

## 2. Reaction network — stoichiometry

Methods are reactions, the space is species space, and what stays open is what no
reaction can change. Hand check: r3 = r1 − 2·r2, so the three reactions span 2
dimensions, and 4 − 2 = 2 conservation laws remain — the H and O atom balances.

In [4]:
c = crn_obj(['H2','O2','H2O','H2O2'], {
    'r1 2H2+O2->2H2O'  : ({'H2':2,'O2':1}, {'H2O':2}),
    'r2 H2+O2->H2O2'   : ({'H2':1,'O2':1}, {'H2O2':1}),
    'r3 2H2O2->2H2O+O2': ({'H2O2':2},      {'H2O':2,'O2':1})}, name="hydrogen / oxygen")
print(c.report())

OBJECT  hydrogen / oxygen    over Q    dim 4    goal: saturate the space
   r1 2H2+O2->2H2O            1 reactions  rank   1   leaves   3 open
   r2 H2+O2->H2O2             1 reactions  rank   1   leaves   3 open
   r3 2H2O2->2H2O+O2          1 reactions  rank   1   leaves   3 open
   ALL                                     rank   2   leaves 2 open
   NO vocabulary in this object closes it


`NO vocabulary in this object closes it` is the correct answer, not a failure. A
reaction network cannot saturate species space — that is what a conservation law is. The
open dimension here is permanent, and this is the one place the report is ambiguous: an
open dimension can mean *no method in the catalogue reaches it* or *no method ever could*.
The tool returns the same line for both and the modeller must know which.

## 3. BBP — residue classes of a series

The space is the coefficient space over residues 1..8 of the base-16 generators
S_r = Σ_k 16^(−k)/(8k+r). A constant is expressible from a set of residues exactly when
its coefficient vector lies in their span. Here the target is

π = 4·S₁ − 2·S₄ − S₅ − S₆,  i.e. (4, 0, 0, −2, −1, −1, 0, 0).

In [5]:
b = bbp_obj({'odd residues {1,3,5,7}':[1,3,5,7], 'even residues {2,4,6,8}':[2,4,6,8],
             'residues {1,5}':[1,5], 'residues {4,6}':[4,6]},
            target=[4,0,0,-2,-1,-1,0,0])
print(b.report())

OBJECT  BBP base-16, degree 1    over Q    dim 8    goal: reach the target
   odd residues {1,3,5,7}     4 residues   rank   4   leaves   1 open
   even residues {2,4,6,8}     4 residues   rank   4   leaves   1 open
   residues {1,5}             2 residues   rank   2   leaves   1 open
   residues {4,6}             2 residues   rank   2   leaves   1 open
   ALL                                     rank   8   CLOSES
   minimal closing vocabularies: odd residues {1,3,5,7}+even residues {2,4,6,8}; odd residues {1,3,5,7}+residues {4,6}; even residues {2,4,6,8}+residues {1,5}; residues {1,5}+residues {4,6}


The minimal vocabularies include `residues {1,5} + residues {4,6}` — which is
{1,4,5,6}, π's actual BBP support. The tool recovered the support by completion, without
being told it.

## 4. SHA-256's linear layer — over GF(2)

The four σ/Σ functions are GF(2)-linear maps on 32 bits. No claim is made here about the
compression function, which is not linear; this is only its linear part, and the question
is the ordinary one: what does each map reach.

In [6]:
s = sha_obj()
print(s.report())

OBJECT  SHA-256 bit-mixing (GF(2)-linear part)    over GF(2)    dim 32    goal: saturate the space
   sigma0                    32 rows       rank  32   leaves   0 open
   sigma1                    32 rows       rank  31   leaves   1 open
   Sigma0                    32 rows       rank  32   leaves   0 open
   Sigma1                    32 rows       rank  32   leaves   0 open
   ALL                                     rank  32   CLOSES
   minimal closing vocabularies: sigma0; Sigma0; Sigma1


σ₀, Σ₀ and Σ₁ are invertible. **σ₁ has rank 31** — it loses one dimension, so one
GF(2) functional of the input is unreadable through it. That is the same report as a
missing cell family in an arena and a conservation law in a reaction network, in a third
domain and over a different field.

## The shared shape

| object | space | methods | what stays open |
|---|---|---|---|
| token arena | invariant cochains | cell families | unreachable relation directions |
| reaction network | species | reactions | conservation laws |
| BBP | residue coefficients | residue classes | the part of the constant unreached |
| SHA linear layer | GF(2)³² | σ/Σ maps | the kernel |

Same five calls, same three possible answers: closes, closes with a named completion, or
leaves an open dimension with an empty candidate list. The last answer is the useful one
and it is the one a framework that only explains successes can never return.